## Growth Predictor Model Training

Forecasts the growth rate of an asset by 1 year, and is designed to provide a float value which fits directly into the Intrinsic Value Analyzer model's input data.

Collected data:
- Tickers, and end-of-year share prices from 2019 till 2023
- model trained on 4 CAGR figures, to forecast the 5th CAGR, next years
- model test data (labels) will come from 2024 end-of-year price
- label calculated from 2023 - 2024 price change or CAGR

#### Imports

In [20]:
from os import path
from csv import DictReader

import logging
import pandas as pd
import yfinance as yf

#### Loading Data from API

In [21]:
data_file = path.join("..", "data")
snp500_path = path.join(data_file, "constituents.csv")
nasdaq_path = path.join(data_file, "nasdaq-listed.csv")
output_path = path.join(data_file, "tickers.csv")
collected_tks = []
existing_tks = set()


with open(snp500_path, mode="r", encoding="utf-8") as f:
    reader = DictReader(f)
    for row in reader:
        ticker = row["Symbol"].strip().upper()
        if ticker and ticker not in existing_tks:
            collected_tks.append(ticker)
            existing_tks.add(ticker)


nasdaq_added_count = 0
with open(nasdaq_path, mode="r", encoding="utf-8") as f:
    reader = DictReader(f)
    for row in reader:
        if nasdaq_added_count >= 500: 
            break
        ticker = row["Symbol"].strip().upper()
        if ticker and ticker not in existing_tks:
            collected_tks.append(ticker)
            existing_tks.add(ticker)
            nasdaq_added_count += 1


print(f"Tickers loaded from {snp500_path} and {nasdaq_path}.")

Tickers loaded from ..\data\constituents.csv and ..\data\nasdaq-listed.csv.


In [22]:
logger = logging.getLogger('yfinance')
logger.setLevel(logging.CRITICAL)
all_price_records = []


try:
    data = yf.download(collected_tks, start="2019-01-09", 
                       end="2025-01-08", group_by='column', progress=False)
    closing = data['Close'] if isinstance(data.columns, pd.MultiIndex) else data
    closing.index = pd.to_datetime(closing.index)
    end_prices = closing.groupby(closing.index.year).last()
    final_df = end_prices.T
    final_df.reset_index(inplace=True)
    final_df.rename(columns={'index': 'Ticker'}, inplace=True)
    final_df.columns = [str(col) for col in final_df.columns]
    columns_order = ['Ticker', '2019', '2020', '2021', '2022', '2023', '2024']
    final_df = final_df.reindex(columns=columns_order)
    final_df.dropna(subset=columns_order, inplace=True)
    final_df.to_csv(output_path, index=False)
    print(f"Saved {len(final_df)} processed tickers to {output_path}")
    print(f"Also added {nasdaq_added_count} tickers from NASDAQ.")
except Exception as e:
    print(f"Error downloading data for multiple tickers: {e}")

Saved 680 processed tickers to ..\data\tickers.csv
Also added 500 tickers from NASDAQ.


#### Calculating CAGR and Scaling ML Data

In [ ]:
# TODO